# Looker Query Watchdog — Interactive Dev Notebook

This notebook is an interactive, run-by-hand version of the `check_running_queries` Cloud Function described in `watchdog.txt`. It uses the same logic (thresholds, source filtering, alerting, optional auto-kill) so it can be used to:

- Verify Looker API credentials and connectivity before deploying anything.
- Inspect currently running queries in a DataFrame.
- Prototype/tune alert thresholds and webhook formatting.
- Test the kill-query path safely, one query at a time, before trusting an unattended Cloud Function to do it.

**This notebook does not run continuously.** It only checks running queries at the moment you execute the cells. For real-time, unattended monitoring (checking every 2-5 minutes, 24/7) the same logic needs to run on a schedule — see the deployment steps for the Cloud Function + Cloud Scheduler pattern.

## Credentials

This notebook authenticates via a `looker.ini` file placed next to this notebook (the standard `looker_sdk` config format). A template has already been created at `looker.ini` — open it and fill in your real `base_url`, `client_id`, and `client_secret` before running the SDK init cell below. Nothing here reads or stores credentials any other way.

In [ ]:
%pip install --quiet looker-sdk requests pandas

In [ ]:
import datetime
import os

import pandas as pd
import requests

import looker_sdk
from looker_sdk import models40

## Configuration

Mirrors the environment variables used by the `main.py` Cloud Function in `watchdog.txt`. Adjust thresholds here as needed — nothing else in the notebook needs to change.

- `INI_FILE` points at the `looker.ini` you filled in with real credentials.
- `ENABLE_AUTO_KILL` stays `False` until you have run this notebook in alert-only mode long enough to trust the thresholds (per the doc's own recommendation: 1-2 weeks alert-only before enabling auto-kill).

In [ ]:
# Path to the looker.ini config file (created alongside this notebook).
INI_FILE = "looker.ini"

# Alert / kill thresholds, in minutes of elapsed in-flight runtime.
ALERT_THRESHOLD_MINUTES = 15
KILL_THRESHOLD_MINUTES = 60

# Auto-kill is OFF by default. See the "Auto-Kill" section below before enabling.
ENABLE_AUTO_KILL = False

# How often you intend to re-run this notebook / how often the deployed
# Cloud Function would run. Used only for the alert de-duplication window.
SCHEDULE_INTERVAL_MINUTES = 5

# Optional: Slack / Google Chat / MS Teams incoming webhook URL.
# Leave blank to just print alerts instead of posting them anywhere.
WEBHOOK_URL = ""

# Sources that must NEVER be flagged or killed by this watchdog.
PROTECTED_SOURCES = {
    "regenerator",    # Persistent Derived Table (PDT) builds
    "alerts",         # Looker scheduled alerts
    "scheduled_task", # Looker scheduled reports & deliveries
    "scheduler",
}

## Initialize the Looker SDK

Reads credentials from `looker.ini` (via `config_file=`) rather than environment variables. `sdk.me()` is a lightweight sanity check — if this fails, double check `looker.ini`'s `base_url` (no `:19999` on Looker Core), `client_id`, and `client_secret`.

In [ ]:
import pathlib

# LOOKERSDK_* environment variables silently override looker.ini if present.
# Clear any stray ones (e.g. left over from following a different guide)
# so looker.ini is always the single source of truth here.
for _env_key in list(os.environ):
    if _env_key.startswith("LOOKERSDK_"):
        print(f"Removing pre-existing env var that would override looker.ini: {_env_key}")
        del os.environ[_env_key]

ini_path = pathlib.Path(INI_FILE).resolve()
print(f"Reading Looker credentials from: {ini_path}")
assert ini_path.exists(), (
    f"looker.ini not found at {ini_path} -- the notebook's working directory "
    "doesn't match where looker.ini lives. Check os.getcwd()."
)

sdk = looker_sdk.init40(config_file=str(ini_path))

me = sdk.me()
print(f"Authenticated as: {me.display_name} <{me.email}>")

## Fetch Currently Running Queries

Calls `GET /api/4.0/running_queries` (`sdk.all_running_queries()`) and loads the raw result into a DataFrame purely for easy visual inspection.

In [ ]:
active_queries = sdk.all_running_queries()

raw_df = pd.DataFrame([
    {
        "query_task_id": q.query_task_id,
        "source": q.source,
        "slug": q.slug,
        "user_display_name": q.user.display_name if q.user else None,
        "connection_name": q.connection_name,
        "created_at": q.created_at,
        "status": q.status,
    }
    for q in active_queries
])

print(f"{len(active_queries)} total running queries")
raw_df

## Filter & Compute Elapsed Runtime

Same logic as `main.py`:
- Skip anything in `PROTECTED_SOURCES` (PDT regenerators, scheduled alerts/reports).
- Keep only interactive `dashboard` / `explore` queries.
- Compute `elapsed_minutes = now_utc - created_at`, since Looker's own `runtime` field is only populated after a query *finishes* (see FAQ in `watchdog.txt`).

In [ ]:
now_utc = datetime.datetime.now(datetime.timezone.utc)


def candidate_queries(queries):
    """Yields (query, elapsed_minutes) for interactive, non-protected, in-flight queries."""
    for q in queries:
        source = (q.source or "").lower()

        if source in PROTECTED_SOURCES:
            continue
        if source not in ("dashboard", "explore"):
            continue
        if not q.created_at:
            continue

        clean_timestamp = q.created_at.replace("Z", "+00:00")
        created_dt = datetime.datetime.fromisoformat(clean_timestamp)
        elapsed_minutes = (now_utc - created_dt).total_seconds() / 60.0

        yield q, elapsed_minutes


candidates = list(candidate_queries(active_queries))

candidates_df = pd.DataFrame([
    {
        "query_task_id": q.query_task_id or "N/A",
        "source": (q.source or "").capitalize(),
        "user_display_name": q.user.display_name if q.user and q.user.display_name else "Unknown User",
        "slug": q.slug or "N/A",
        "connection_name": q.connection_name,
        "elapsed_minutes": round(elapsed_minutes, 1),
    }
    for q, elapsed_minutes in candidates
]).sort_values("elapsed_minutes", ascending=False, ignore_index=True) if candidates else pd.DataFrame()

candidates_df

## Notification Helper

Same behavior as `main.py`: posts to `WEBHOOK_URL` if set (Slack-compatible `{"text": ...}` payload works for Slack and Google Chat incoming webhooks), otherwise just prints a dry-run notice.

In [ ]:
def send_notification(message: str) -> None:
    """Sends notification payload to configured webhook, or prints a dry-run notice."""
    if not WEBHOOK_URL:
        print(f"[DRY RUN NOTICE]:\n{message}\n")
        return

    try:
        response = requests.post(WEBHOOK_URL, json={"text": message}, timeout=10)
        response.raise_for_status()
    except Exception as err:
        print(f"Failed to deliver webhook alert: {err}")

## Alert Tier

Flags queries whose elapsed runtime falls in `[ALERT_THRESHOLD_MINUTES, ALERT_THRESHOLD_MINUTES + SCHEDULE_INTERVAL_MINUTES)`. That window (rather than a simple `>=` check) is what the doc calls "alert de-duplication" — it's designed so that, once deployed on a fixed schedule, each query is only alerted on once as it *enters* the threshold, instead of re-alerting on every subsequent poll. Run manually, this window just means "flag queries that just crossed the alert threshold since the last time you checked."

In [ ]:
flagged_queries = 0

for q, elapsed_minutes in candidates:
    if ALERT_THRESHOLD_MINUTES <= elapsed_minutes < (ALERT_THRESHOLD_MINUTES + SCHEDULE_INTERVAL_MINUTES):
        flagged_queries += 1

        user_display_name = q.user.display_name if q.user and q.user.display_name else "Unknown User"
        query_slug = q.slug or "N/A"

        alert_msg = (
            f"⚠️ *Looker Core: Long-Running Query Detected*\n"
            f"• *User:* {user_display_name}\n"
            f"• *Source:* {(q.source or '').capitalize()}\n"
            f"• *Current Runtime:* {elapsed_minutes:.1f} minutes\n"
            f"• *Connection:* {q.connection_name}\n"
            f"• *Query Slug:* `{query_slug}`\n"
            f"• *Action Required:* Review dashboard filters or cancel task from Admin → Queries."
        )
        send_notification(alert_msg)

print(f"Flagged {flagged_queries} long-running quer{'y' if flagged_queries == 1 else 'ies'}.")

## ⚠️ Auto-Kill Tier — Destructive, Off By Default

**Read this before enabling.** Setting `ENABLE_AUTO_KILL = True` above will make the next cell actually **cancel running queries** on the warehouse via `sdk.kill_query()`, for any non-protected `dashboard`/`explore` query running longer than `KILL_THRESHOLD_MINUTES`. This is irreversible for that query run.

Per `watchdog.txt`'s own guidance:
- Run in alert-only mode (`ENABLE_AUTO_KILL = False`) for 1-2 weeks first to confirm your thresholds don't false-positive on legitimately long dashboards.
- Never kill `regenerator` (PDT builds), `alerts`, `scheduled_task`, or `scheduler` sources — the filtering above already protects those, but double check before changing `PROTECTED_SOURCES`.
- The service account used in `looker.ini` needs the `cancel_queries` permission for this to succeed.

The cell below is a no-op unless you explicitly set `ENABLE_AUTO_KILL = True` in the configuration cell and re-run from there.

In [ ]:
killed_queries = 0

if not ENABLE_AUTO_KILL:
    print("ENABLE_AUTO_KILL is False -- skipping kill checks (alert-only mode).")
else:
    for q, elapsed_minutes in candidates:
        task_id = q.query_task_id
        if elapsed_minutes < KILL_THRESHOLD_MINUTES or not task_id:
            continue

        try:
            sdk.kill_query(task_id)
            killed_queries += 1
            kill_notice = (
                f"🛑 *Looker Core: Auto-Killed Runaway Query*\n"
                f"• *User:* {q.user.display_name if q.user and q.user.display_name else 'Unknown User'}\n"
                f"• *Source:* {(q.source or '').capitalize()}\n"
                f"• *Runtime:* {elapsed_minutes:.1f} minutes (Ceiling: {KILL_THRESHOLD_MINUTES}m)\n"
                f"• *Connection:* {q.connection_name}\n"
                f"• *Query Task ID:* `{task_id}`"
            )
            send_notification(kill_notice)
        except Exception as err:
            print(f"Failed to kill query task {task_id}: {err}")

    print(f"Killed {killed_queries} runaway quer{'y' if killed_queries == 1 else 'ies'}.")

## Summary

Same shape as the JSON the Cloud Function returns to Cloud Scheduler on each invocation.

In [ ]:
summary = {
    "status": "success",
    "inspected_queries": len(active_queries),
    "flagged_alerts": flagged_queries,
    "killed_queries": killed_queries,
}
summary

## Continuous Test Loop (Local Polling)

For exactly this situation: embed users are about to run dashboards for the next 5-6 minutes and you want to *watch* the watchdog catch their queries, without a slow/ad-hoc query and without manually re-running cells 5-10 over and over.

This cell polls `all_running_queries()` every `POLL_EVERY_SECONDS`, for a total of `TEST_DURATION_MINUTES`, and prints every dashboard/explore query it sees along with elapsed runtime as it happens.

**Before running this:** your `ALERT_THRESHOLD_MINUTES` is set to `15` above. A 5-6 minute test will never reach that. To actually see an alert fire during a short test window, go back to the configuration cell and temporarily lower it (e.g. `ALERT_THRESHOLD_MINUTES = 1`), then re-run that cell before running this one. Put it back to `15` afterward.

Requires cells 1-4 (setup + SDK init) and the config / filter / notification-helper cells above to have already run — this reuses `candidate_queries()` and `send_notification()` defined there.

- Each query is alerted **at most once per run of this loop** (tracked by `query_task_id`) so a query that's still over threshold on the next poll doesn't spam the webhook again.
- Stop early anytime with the notebook's Interrupt/Stop button.
- This is for manual testing only — it stops the moment this cell finishes or the notebook closes. It does not replace the scheduled Cloud Function in production.

In [ ]:
import time

TEST_DURATION_MINUTES = 6   # how long to keep polling
POLL_EVERY_SECONDS = 2      # how often to check running queries

test_end = datetime.datetime.now(datetime.timezone.utc) + datetime.timedelta(minutes=TEST_DURATION_MINUTES)
already_alerted_task_ids = set()
poll_count = 0
loop_flagged = 0
loop_killed = 0

print(f"Polling every {POLL_EVERY_SECONDS}s for {TEST_DURATION_MINUTES} minute(s). Interrupt the cell to stop early.")

while datetime.datetime.now(datetime.timezone.utc) < test_end:
    poll_count += 1
    now_utc = datetime.datetime.now(datetime.timezone.utc)
    active_queries = sdk.all_running_queries()
    candidates = list(candidate_queries(active_queries))

    print(f"\n--- Poll #{poll_count} @ {now_utc:%H:%M:%S} UTC | {len(active_queries)} running total | {len(candidates)} dashboard/explore candidate(s) ---")

    for q, elapsed_minutes in candidates:
        task_id = q.query_task_id or "N/A"
        user_display_name = q.user.display_name if q.user and q.user.display_name else "Unknown User"
        print(f"  - {user_display_name} | {(q.source or '').capitalize()} | {elapsed_minutes:.1f} min elapsed | task {task_id}")

        if ENABLE_AUTO_KILL and elapsed_minutes >= KILL_THRESHOLD_MINUTES and task_id != "N/A":
            try:
                sdk.kill_query(task_id)
                loop_killed += 1
                already_alerted_task_ids.add(task_id)
                send_notification(
                    f"🛑 *Looker Core: Auto-Killed Runaway Query*\n"
                    f"• *User:* {user_display_name}\n"
                    f"• *Source:* {(q.source or '').capitalize()}\n"
                    f"• *Runtime:* {elapsed_minutes:.1f} minutes (Ceiling: {KILL_THRESHOLD_MINUTES}m)\n"
                    f"• *Connection:* {q.connection_name}\n"
                    f"• *Query Task ID:* `{task_id}`"
                )
                continue
            except Exception as err:
                print(f"    Failed to kill query task {task_id}: {err}")

        if elapsed_minutes >= ALERT_THRESHOLD_MINUTES and task_id not in already_alerted_task_ids:
            already_alerted_task_ids.add(task_id)
            loop_flagged += 1
            send_notification(
                f"⚠️ *Looker Core: Long-Running Query Detected*\n"
                f"• *User:* {user_display_name}\n"
                f"• *Source:* {(q.source or '').capitalize()}\n"
                f"• *Current Runtime:* {elapsed_minutes:.1f} minutes\n"
                f"• *Connection:* {q.connection_name}\n"
                f"• *Query Slug:* `{q.slug or 'N/A'}`\n"
                f"• *Action Required:* Review dashboard filters or cancel task from Admin → Queries."
            )

    if datetime.datetime.now(datetime.timezone.utc) < test_end:
        time.sleep(POLL_EVERY_SECONDS)

print(f"\nDone. {poll_count} poll(s) over {TEST_DURATION_MINUTES} minute(s) -- flagged {loop_flagged}, killed {loop_killed}.")

## Next Step: Going to Production

Everything above is deliberately structured to map 1:1 onto `main.py` in `watchdog.txt` — the same config knobs, the same filtering logic, the same alert/kill message formats. To productionize:

1. Re-run this notebook periodically by hand whenever you want a point-in-time check, **or**
2. Deploy the logic as the Cloud Function described in `watchdog.txt` (Cloud Scheduler → Cloud Function → Looker API), which runs unattended every 2-5 minutes. That version reads credentials from environment variables / Secret Manager instead of `looker.ini`, since there's no interactive session to point at a local file — but the query/filter/alert/kill logic is identical to what's in this notebook.

See the deployment walkthrough for the exact `gcloud` commands.